# Apache Spark Learning Notebook

This comprehensive notebook covers Apache Spark from fundamentals to advanced and latest features. Learn distributed computing, data processing, and real-time analytics.

**Topics Covered:**
1. Setting Up Spark Environment
2. Understanding Spark Architecture and RDDs
3. Working with DataFrames and Datasets
4. Data Loading and Writing
5. DataFrame Operations and Transformations
6. SQL Queries on Spark DataFrames
7. Aggregations and Grouping
8. Joins and Set Operations
9. Window Functions
10. User Defined Functions (UDFs)
11. Performance Optimization and Tuning
12. Structured Streaming
13. Machine Learning with MLlib
14. Graph Processing with GraphX
15. Delta Lake and Data Quality

**Prerequisites:**
- Python 3.7+
- Java 8+
- Apache Spark 3.2+ (or latest version)


## 1. Setting Up Spark Environment

Install and configure Apache Spark, create a SparkSession, and verify the installation. Initialize Spark with appropriate configurations for local and distributed environments.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark import SparkContext
import pyspark.sql.functions as F

# Create a SparkSession - the entry point for Spark functionality
spark = SparkSession.builder \
    .appName("Spark Learning") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

# Get SparkContext
sc = spark.sparkContext

# Set log level to INFO (reduce verbosity)
sc.setLogLevel("WARN")

# Verify installation
print(f"Spark Version: {spark.version}")
print(f"Python Version: {sc.pythonVer}")
print(f"Application Name: {sc.appName}")
print(f"Master: {sc.master}")
print(f"Number of Partitions: {sc.defaultParallelism}")
print("\nSpark is successfully configured!")


## 2. Understanding Spark Architecture and RDDs

Learn Spark's distributed computing architecture, RDD fundamentals, transformations, actions, and lazy evaluation. Understand partitioning and data distribution.

### Key Concepts:
- **RDD (Resilient Distributed Dataset)**: Immutable, distributed collection of objects
- **Transformations**: Lazily evaluated operations (map, filter, flatMap, etc.)
- **Actions**: Operations that return results to driver or write data (collect, count, save, etc.)
- **Lazy Evaluation**: Transformations are not executed until an action is called
- **Partitions**: Data is divided into partitions for parallel processing


In [ ]:
# Creating RDDs
# Method 1: Parallelize a collection
rdd_from_list = sc.parallelize([1, 2, 3, 4, 5], numPartitions=2)
print("RDD created from list:", rdd_from_list.collect())
print("Number of partitions:", rdd_from_list.getNumPartitions())
print("RDD count:", rdd_from_list.count())

# Method 2: Transform existing RDD
# Transformations (Lazy - not executed until action is called)
rdd_mapped = rdd_from_list.map(lambda x: x * 2)  # Transformation
print("\nAfter map (transformation applied):", rdd_mapped.collect())  # Action triggers execution

# Filter transformation
rdd_filtered = rdd_from_list.filter(lambda x: x > 2)
print("After filter:", rdd_filtered.collect())

# FlatMap transformation
rdd_flat = rdd_from_list.flatMap(lambda x: [x, x*2])
print("After flatMap:", rdd_flat.collect())

# Map on strings
string_rdd = sc.parallelize(["hello world", "spark rdd", "distributed computing"])
word_rdd = string_rdd.flatMap(lambda x: x.split())
print("Words from flatMap:", word_rdd.collect())

# Distinct transformation
rdd_with_dupes = sc.parallelize([1, 2, 2, 3, 3, 3, 4, 5, 5])
rdd_distinct = rdd_with_dupes.distinct()
print("After distinct:", rdd_distinct.collect())


## 3. Working with DataFrames and Datasets

Create DataFrames from various sources, understand schemas, and explore DataFrame APIs. Learn the differences between RDDs, DataFrames, and Datasets.

### Key Concepts:
- **DataFrame**: Distributed collection with named columns (similar to SQL table or Pandas DataFrame)
- **Schema**: Structure defining column names and data types
- **Catalyst Optimizer**: Optimizes DataFrame queries automatically
- **Dataset**: Type-safe, object-oriented interface (mainly for Scala/Java)


In [ ]:
# Creating DataFrames
# Method 1: From data with explicit schema
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("salary", DoubleType(), True)
])

data = [
    (1, "Alice", 30, 50000.0),
    (2, "Bob", 25, 45000.0),
    (3, "Charlie", 35, 60000.0),
    (4, "David", 28, 48000.0)
]

df_people = spark.createDataFrame(data, schema)
print("DataFrame created with explicit schema:")
df_people.show()
df_people.printSchema()

# Method 2: From data with inferred schema
data_inferred = [("Alice", 30), ("Bob", 25), ("Charlie", 35)]
df_simple = spark.createDataFrame(data_inferred, ["name", "age"])
print("\nDataFrame with inferred schema:")
df_simple.show()
df_simple.printSchema()

# DataFrame info
print(f"\nDataFrame shape: {df_people.count()} rows, {len(df_people.columns)} columns")
print(f"Column names: {df_people.columns}")
print(f"Data types: {df_people.dtypes}")


## 4. Data Loading and Writing

Load data from CSV, JSON, Parquet, ORC, and other formats. Write DataFrames to different storage systems with various write modes.

### Write Modes:
- **overwrite**: Overwrite existing file
- **append**: Append to existing file
- **ignore**: Silently ignore the operation
- **error**: Throw an error if file exists (default)


In [ ]:
import os

# Create sample data for demonstration
sample_data = [
    (1, "Alice", 30, "Engineering", 50000),
    (2, "Bob", 25, "Sales", 45000),
    (3, "Charlie", 35, "Management", 60000),
    (4, "David", 28, "Engineering", 52000),
    (5, "Eve", 32, "Sales", 48000)
]

df_employees = spark.createDataFrame(sample_data, ["id", "name", "age", "department", "salary"])

# Create output directory
output_dir = "spark_data"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Writing to different formats
print("Writing DataFrame to different formats...\n")

# Write to CSV
csv_path = f"{output_dir}/employees.csv"
df_employees.write.mode("overwrite").csv(csv_path, header=True)
print(f"✓ Written to CSV: {csv_path}")

# Write to JSON
json_path = f"{output_dir}/employees.json"
df_employees.write.mode("overwrite").json(json_path)
print(f"✓ Written to JSON: {json_path}")

# Write to Parquet
parquet_path = f"{output_dir}/employees.parquet"
df_employees.write.mode("overwrite").parquet(parquet_path)
print(f"✓ Written to Parquet: {parquet_path}")

# Reading from different formats
print("\nReading data from different formats...\n")

# Read from CSV
df_csv = spark.read.csv(csv_path, header=True, inferSchema=True)
print("Data from CSV:")
df_csv.show()

# Read from JSON
df_json = spark.read.json(json_path)
print("Data from JSON:")
df_json.show()

# Read from Parquet
df_parquet = spark.read.parquet(parquet_path)
print("Data from Parquet:")
df_parquet.show()


## 5. DataFrame Operations and Transformations

Perform column selection, filtering, map, flatMap, distinct, drop duplicates, and other transformations. Work with null values and data type conversions.


In [ ]:
# Column selection
print("=== Column Selection ===")
df_selected = df_employees.select("name", "age", "salary")
df_selected.show()

# Selection with new column names
df_renamed = df_employees.select(
    F.col("name").alias("employee_name"),
    F.col("age").alias("years_old"),
    F.col("salary").alias("annual_salary")
)
print("\nRenamedcolumns:")
df_renamed.show()

# Filtering
print("\n=== Filtering ===")
df_filtered = df_employees.filter(df_employees.age > 30)
print("Employees with age > 30:")
df_filtered.show()

# Multiple conditions
df_filtered2 = df_employees.filter((F.col("age") > 25) & (F.col("salary") > 48000))
print("Employees with age > 25 AND salary > 48000:")
df_filtered2.show()

# Add new columns
print("\n=== Adding Columns ===")
df_with_new_col = df_employees.withColumn("salary_raise", F.col("salary") * 1.1)
print("Added salary_raise column:")
df_with_new_col.show()

# Drop columns
print("\n=== Dropping Columns ===")
df_dropped = df_employees.drop("department")
print("After dropping department column:")
df_dropped.show()

# Type conversion
print("\n=== Type Conversions ===")
df_converted = df_employees.withColumn("age_string", F.col("age").cast("string"))
df_converted.printSchema()

# Distinct values
print("\n=== Distinct Values ===")
df_depts = df_employees.select("department").distinct()
print("Unique departments:")
df_depts.show()


## 6. SQL Queries on Spark DataFrames

Convert DataFrames to temporary views and write SQL queries. Understand Catalyst optimizer and leverage SQL for complex analytics.

### Benefits:
- Familiar SQL syntax for data analysis
- Catalyst automatically optimizes queries
- Can combine DataFrame API and SQL seamlessly


In [ ]:
# Create temporary views
df_employees.createOrReplaceTempView("employees")

# Simple SELECT query
print("=== Simple SELECT Query ===")
result1 = spark.sql("SELECT name, age, salary FROM employees WHERE age > 25")
result1.show()

# Aggregation with GROUP BY
print("\n=== GROUP BY Query ===")
result2 = spark.sql("""
    SELECT 
        department, 
        COUNT(*) as employee_count, 
        AVG(salary) as avg_salary,
        MAX(salary) as max_salary
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
""")
result2.show()

# JOIN example (self-join for demonstration)
df_employees.createOrReplaceTempView("emp1")
df_employees.createOrReplaceTempView("emp2")

print("\n=== JOIN Query ===")
result3 = spark.sql("""
    SELECT 
        e1.name as name1,
        e2.name as name2,
        e1.department
    FROM emp1 e1
    JOIN emp2 e2 ON e1.department = e2.department
    WHERE e1.id < e2.id
    LIMIT 5
""")
result3.show()

# Check Catalyst execution plan
print("\n=== Catalyst Execution Plan ===")
result1.explain(extended=True)


## 7. Aggregations and Grouping

Perform aggregations using agg(), groupBy(), and aggregate functions. Implement custom aggregations and multi-level grouping.

### Common Aggregate Functions:
- count, sum, avg, min, max, stddev, variance
- first, last, collect_list, collect_set


In [ ]:
# Basic aggregations
print("=== Basic Aggregations ===")
agg_result = df_employees.agg(
    F.count("id").alias("total_employees"),
    F.sum("salary").alias("total_salary"),
    F.avg("salary").alias("avg_salary"),
    F.min("salary").alias("min_salary"),
    F.max("salary").alias("max_salary"),
    F.stddev("salary").alias("salary_stddev")
)
agg_result.show()

# GroupBy operations
print("\n=== GroupBy Operations ===")
dept_agg = df_employees.groupBy("department").agg(
    F.count("id").alias("count"),
    F.avg("salary").alias("avg_salary"),
    F.max("age").alias("max_age")
).orderBy("avg_salary", ascending=False)
dept_agg.show()

# Multiple grouping columns
print("\n=== Multi-level Grouping ===")
df_with_group = df_employees.withColumn("salary_range", 
    F.when(F.col("salary") < 45000, "Low")
     .when(F.col("salary") < 55000, "Medium")
     .otherwise("High"))

multi_group = df_with_group.groupBy("department", "salary_range").agg(
    F.count("id").alias("count"),
    F.avg("age").alias("avg_age")
)
multi_group.show()

# Collect aggregations
print("\n=== Collect Aggregations ===")
collect_agg = df_employees.groupBy("department").agg(
    F.collect_list("name").alias("employees"),
    F.collect_set("age").alias("unique_ages")
)
collect_agg.show(truncate=False)


## 8. Joins and Set Operations

Execute inner, left, right, outer, cross, and broadcast joins. Perform union, intersect, and except operations.

### Join Types:
- **inner**: Only matching records from both DataFrames
- **left**: All records from left, matching from right
- **right**: All records from right, matching from left
- **outer**: All records from both DataFrames
- **cross**: Cartesian product of both DataFrames


In [ ]:
# Create sample datasets for joins
df_admin = spark.createDataFrame([
    (1, "A1"),
    (2, "A2"),
    (3, "A3"),
    (5, "A5")
], ["emp_id", "admin_code"])

df_hr = spark.createDataFrame([
    (1, "H1"),
    (2, "H2"),
    (4, "H4"),
    (6, "H6")
], ["emp_id", "hr_code"])

# INNER JOIN
print("=== INNER JOIN ===")
inner_join = df_admin.join(df_hr, on="emp_id", how="inner")
inner_join.show()

# LEFT JOIN
print("\n=== LEFT JOIN ===")
left_join = df_admin.join(df_hr, on="emp_id", how="left")
left_join.show()

# RIGHT JOIN
print("\n=== RIGHT JOIN ===")
right_join = df_admin.join(df_hr, on="emp_id", how="right")
right_join.show()

# OUTER JOIN
print("\n=== OUTER JOIN ===")
outer_join = df_admin.join(df_hr, on="emp_id", how="outer")
outer_join.show()

# Broadcast Join (optimization for small tables)
print("\n=== BROADCAST JOIN ===")
from pyspark.sql.functions import broadcast
broadcast_join = df_employees.join(
    broadcast(df_admin), 
    df_employees.id == df_admin.emp_id, 
    how="left"
)
broadcast_join.select("name", "department", "admin_code").show()

# Set Operations
print("\n=== UNION ===")
df1 = spark.createDataFrame([(1, "A"), (2, "B"), (3, "C")], ["id", "value"])
df2 = spark.createDataFrame([(3, "C"), (4, "D")], ["id", "value"])

union_result = df1.union(df2)
print("UNION (with duplicates):")
union_result.show()

print("\nUNION ALL (distinct):")
union_all = df1.unionByName(df2, allowMissingColumns=False).distinct()
union_all.show()

# INTERSECT
print("\n=== INTERSECT ===")
intersect_result = df1.intersect(df2)
intersect_result.show()

# EXCEPT
print("\n=== EXCEPT ===")
except_result = df1.exceptAll(df2)
except_result.show()


## 9. Window Functions

Use window functions for ranking, lead/lag operations, and running aggregations. Define window specifications and frame boundaries.

### Window Function Categories:
- **Ranking**: rank(), row_number(), dense_rank(), percent_rank()
- **Analytic**: lead(), lag(), first_value(), last_value()
- **Aggregate**: sum(), avg(), min(), max() over window


In [ ]:
from pyspark.sql.window import Window

# Ranking functions
print("=== Ranking Functions ===")
window_rank = Window.partitionBy("department").orderBy(F.desc("salary"))

rank_df = df_employees.withColumn("rank", F.rank().over(window_rank)) \
                       .withColumn("dense_rank", F.dense_rank().over(window_rank)) \
                       .withColumn("row_number", F.row_number().over(window_rank))
rank_df.select("name", "department", "salary", "rank", "dense_rank", "row_number").show()

# Lead and Lag functions
print("\n=== Lead/Lag Functions ===")
window_order = Window.partitionBy("department").orderBy("salary")

lead_lag_df = df_employees.withColumn("prev_salary", F.lag("salary", 1).over(window_order)) \
                           .withColumn("next_salary", F.lead("salary", 1).over(window_order))
lead_lag_df.select("name", "department", "salary", "prev_salary", "next_salary").show()

# Running aggregations
print("\n=== Running Aggregations ===")
window_running = Window.partitionBy("department") \
                        .orderBy("salary") \
                        .rangeBetween(Window.unboundedPreceding, Window.currentRow)

running_agg = df_employees.withColumn("running_sum", F.sum("salary").over(window_running)) \
                           .withColumn("running_count", F.count("id").over(window_running)) \
                           .withColumn("running_avg", F.avg("salary").over(window_running))
running_agg.select("name", "department", "salary", "running_sum", "running_avg").show()

# Cumulative distribution
print("\n=== Cumulative Distribution ===")
window_cume = Window.partitionBy("department").orderBy("salary")
cume_df = df_employees.withColumn("cume_dist", F.cume_dist().over(window_cume))
cume_df.select("name", "department", "salary", "cume_dist").show()

# Top N per group
print("\n=== Top 2 Highest Salary per Department ===")
top_n = rank_df.filter("rank <= 2").select("name", "department", "salary", "rank")
top_n.show()


## 10. User Defined Functions (UDFs)

Create Python UDFs, understand performance implications, and explore Pandas UDFs for vectorization benefits.

### UDF Types:
- **Python UDF**: Row-by-row processing (slower)
- **Pandas UDF**: Vectorized processing (faster)
- **SQL UDF**: SQL-based user functions


In [ ]:
from pyspark.sql.types import StringType, DoubleType
import pandas as pd

# Python UDF (slower - row-by-row processing)
print("=== Python UDF (Row-by-Row) ===")

@F.udf(returnType=StringType())
def categorize_salary(salary):
    if salary < 46000:
        return "Entry Level"
    elif salary < 52000:
        return "Mid Level"
    else:
        return "Senior Level"

df_with_category = df_employees.withColumn("salary_category", categorize_salary(F.col("salary")))
df_with_category.show()

# Pandas UDF (faster - vectorized processing)
print("\n=== Pandas UDF (Vectorized) ===")

@F.pandas_udf(returnType=DoubleType())
def bonus_calculation(salaries):
    # This processes a batch of rows at once
    return salaries * 0.15

df_with_bonus = df_employees.withColumn("bonus", bonus_calculation(F.col("salary")))
df_with_bonus.select("name", "salary", "bonus").show()

# Pandas UDF returning multiple columns (Series to StructType)
print("\n=== Pandas UDF with Multiple Outputs ===")

@F.pandas_udf(returnType="bonus DOUBLE, commission DOUBLE")
def income_supplements(salaries):
    bonus = salaries * 0.15
    commission = salaries * 0.05
    return pd.DataFrame({"bonus": bonus, "commission": commission})

df_supplements = df_employees.withColumn("supplements", income_supplements(F.col("salary"))) \
                               .select("name", "salary", "supplements.*")
df_supplements.show()

# SQL UDF
print("\n=== SQL UDF ===")
spark.udf.registerFunction("scale_salary", lambda s: s * 1.1, DoubleType())
result_sql_udf = spark.sql("SELECT name, salary, scale_salary(salary) as scaled_salary FROM employees LIMIT 5")
result_sql_udf.show()

# Testing performance
print("\n=== Performance: Python UDF vs Pandas UDF ===")
large_df = spark.range(0, 100000).withColumn("salary", F.col("id") * 100 + 40000)

import time

# Python UDF
start = time.time()
result_python = large_df.withColumn("category", categorize_salary(F.col("salary"))).count()
python_time = time.time() - start

# Pandas UDF
start = time.time()
result_pandas = large_df.withColumn("bonus", bonus_calculation(F.col("salary"))).count()
pandas_time = time.time() - start

print(f"Python UDF time: {python_time:.2f}s")
print(f"Pandas UDF time: {pandas_time:.2f}s")
print(f"Speedup: {python_time/pandas_time:.2f}x")


## 11. Performance Optimization and Tuning

Implement partitioning strategies, optimize shuffles, leverage caching, use broadcast variables, and understand execution plans.

### Optimization Strategies:
- **Caching**: Store intermediate DataFrames in memory
- **Partitioning**: Distribute data efficiently
- **Broadcasting**: Send small DataFrames to all executors
- **Explain Plans**: Understand query execution
- **Column Pruning**: Select only needed columns


In [ ]:
import time

# 1. Caching/Persistence
print("=== Caching ===")
df_large = spark.range(0, 1000000).withColumn("dept", F.col("id") % 10)

# Check cache storage
print(f"Before cache: {spark.catalog.cachedDataFrames()}")

# Cache the DataFrame
df_large.cache()
print(f"After cache: {spark.catalog.cachedDataFrames()}")
df_large.count()  # Trigger caching

# Uncache when done
df_large.unpersist()
print(f"After unpersist: {spark.catalog.cachedDataFrames()}")

# 2. Partitioning
print("\n=== Partitioning ===")
df_partitioned = df_employees.repartition(4, "department")
print(f"Number of partitions: {df_partitioned.rdd.getNumPartitions()}")

# Coalesce (reduce partitions)
df_coalesced = df_partitioned.coalesce(2)
print(f"After coalesce: {df_coalesced.rdd.getNumPartitions()}")

# 3. Broadcasting large variables
print("\n=== Broadcasting ===")
broadcast_lookup = {
    "Engineering": "ENG",
    "Sales": "SAL",
    "Management": "MAN"
}

broadcast_var = sc.broadcast(broadcast_lookup)

@F.udf(returnType=StringType())
def get_dept_code(dept):
    return broadcast_var.value.get(dept, "UNKNOWN")

df_with_codes = df_employees.withColumn("dept_code", get_dept_code(F.col("department")))
df_with_codes.select("name", "department", "dept_code").show()

# 4. Execution Plans
print("\n=== Execution Plans ===")
query_df = df_employees.filter(F.col("salary") > 50000) \
                        .select("name", "salary") \
                        .orderBy("salary")

print("\nPhysical Plan (Optimized):")
query_df.explain(mode="physical")

print("\nLogical Plan:")
query_df.explain(mode="logical")

# 5. Performance metrics
print("\n=== DataFrame Info ===")
print(f"Total rows: {df_employees.count()}")
print(f"Columns: {df_employees.columns}")
print(f"Estimated compression ratio: {df_employees.rdd.getStorageLevel()}")

# Get Spark configuration
print("\n=== Spark Configuration (Sample) ===")
conf_dict = spark.sparkContext.getConf().getAll()
for key, value in sorted(conf_dict)[:10]:
    print(f"{key}: {value}")


## 12. Structured Streaming

Build real-time data pipelines using Structured Streaming. Work with streaming DataFrames, micro-batches, watermarking, and output modes.

### Key Concepts:
- **Streaming DataFrame**: Continues to receive new data
- **Micro-batches**: Process data in small batches
- **Watermarking**: Handle late-arriving data
- **Output Modes**: append, update, complete
- **Triggers**: When to process batches (processingTime, once, continuous)


In [ ]:
# Structured Streaming Example (Memory source for demonstration)
print("=== Structured Streaming Example ===")

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime, timedelta

# Create a streaming DataFrame from socket data
# Note: This is a demonstration. In production, you'd use real sources like Kafka

# For demonstration, we'll use a rate source
rate_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load()

print("Streaming DataFrame schema:")
rate_stream.printSchema()

# Transform streaming data
transformed_stream = rate_stream.select(
    F.col("timestamp"),
    (F.col("value") % 10).alias("value"),
    F.current_timestamp().alias("processed_at")
)

# Aggregation on streaming data
print("\n=== Streaming Aggregation ===")
agg_stream = transformed_stream.groupBy(
    F.window(F.col("timestamp"), "10 seconds"),
    "value"
).count()

# Write streaming results (without starting, as this is for demonstration)
print("Streaming query setup (not executed due to notebook constraints):")
print("Output modes: append, update, complete")
print("Triggers: processingTime, once, continuous")

print("""
# Example query to run in production:
query = agg_stream \\
    .writeStream \\
    .format("console") \\
    .outputMode("update") \\
    .trigger(processingTime="5 seconds") \\
    .start()

query.awaitTermination()
""")

# Watermarking example (conceptual)
print("\n=== Watermarking (Conceptual) ===")
print("""
# Handle late-arriving data
streaming_df_watermarked = transformed_stream \\
    .withWatermark("timestamp", "2 minutes") \\
    .groupBy(F.window(F.col("timestamp"), "5 minutes")) \\
    .agg(F.count("value").alias("count"))
""")


## 13. Machine Learning with MLlib

Use MLlib for classification, regression, clustering, and recommendation systems. Build ML pipelines with transformers and estimators.

### MLlib Components:
- **Transformers**: Convert one DataFrame to another (e.g., Tokenizer, StandardScaler)
- **Estimators**: Learn parameters from data (e.g., LogisticRegression, KMeans)
- **Pipelines**: Combine multiple transformers and estimators
- **Evaluators**: Evaluate model performance


In [ ]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.regression import LinearRegression
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import BinaryClassificationEvaluator, ClusteringEvaluator
from pyspark.ml.stat import Correlation
import numpy as np

print("=== Feature Engineering ===")

# Create sample data for ML
ml_data = spark.createDataFrame([
    (1, 5.1, 3.5, "Setosa"),
    (2, 7.0, 3.2, "Versicolor"),
    (3, 6.3, 3.3, "Virginica"),
    (4, 4.9, 3.0, "Setosa"),
    (5, 6.1, 2.9, "Versicolor")
] * 20, ["id", "feature1", "feature2", "label"])

# String indexing for categorical labels
indexer = StringIndexer(inputCol="label", outputCol="label_index")
indexed_data = indexer.fit(ml_data).transform(ml_data)

# Feature assembly
assembler = VectorAssembler(inputCols=["feature1", "feature2"], outputCol="features")
assembled_data = assembler.transform(indexed_data)

# Feature scaling
scaler = StandardScaler(inputCol="features", outputCol="scaled_features")
scaled_data = scaler.fit(assembled_data).transform(assembled_data)

print("Transformed data:")
scaled_data.select("label", "features", "scaled_features").show(3, truncate=False)

# Split data
train_data, test_data = scaled_data.randomSplit([0.7, 0.3], seed=42)
print(f"\nTrain set: {train_data.count()}, Test set: {test_data.count()}")

# Logistic Regression
print("\n=== Logistic Regression ===")
lr = LogisticRegression(featuresCol="scaled_features", labelCol="label_index", maxIter=10)
lr_model = lr.fit(train_data)

lr_predictions = lr_model.transform(test_data)
print("Predictions:")
lr_predictions.select("label", "prediction", "probability").show(5)

# Random Forest
print("\n=== Random Forest Classification ===")
rf = RandomForestClassifier(featuresCol="scaled_features", labelCol="label_index", numTrees=3)
rf_model = rf.fit(train_data)

rf_predictions = rf_model.transform(test_data)
print("Random Forest predictions:")
rf_predictions.select("label", "prediction").show(5)

# K-Means Clustering
print("\n=== K-Means Clustering ===")
kmeans = KMeans(k=3, seed=42, maxIter=10)
kmeans_model = kmeans.fit(scaled_data)

clustered_data = kmeans_model.transform(scaled_data)
print("Clustered data:")
clustered_data.select("label", "prediction").show(5)

# Model Evaluation
print("\n=== Model Evaluation ===")
evaluator = BinaryClassificationEvaluator(labelCol="label_index", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
print(f"Logistic Regression AUC: {evaluator.evaluate(lr_predictions):.4f}")
print(f"Random Forest AUC: {evaluator.evaluate(rf_predictions):.4f}")

# Pipeline Example
print("\n=== ML Pipeline ===")
pipeline = Pipeline(stages=[
    indexer,
    assembler,
    scaler,
    LogisticRegression(featuresCol="scaled_features", labelCol="label_index", maxIter=10)
])

pipeline_model = pipeline.fit(train_data)
pipeline_predictions = pipeline_model.transform(test_data)
print("Pipeline predictions:")
pipeline_predictions.select("label", "prediction").show(3)


## 14. Graph Processing with GraphX

Create and manipulate graphs, apply graph algorithms, perform graph queries, and optimize graph computations.

### Key Concepts:
- **Vertices**: Nodes with properties
- **Edges**: Connections between vertices with properties
- **Graph Algorithms**: PageRank, triangle counting, shortest path
- **Vertex/Edge RDDs**: Direct access to underlying data structures


In [ ]:
from graphframes import GraphFrame

print("=== Creating a Graph ===")

# Create vertices
vertices = spark.createDataFrame([
    (1, "Alice", "Engineering"),
    (2, "Bob", "Sales"),
    (3, "Charlie", "Engineering"),
    (4, "David", "Management"),
    (5, "Eve", "Sales")
], ["id", "name", "department"])

# Create edges (relationships/connections)
edges = spark.createDataFrame([
    (1, 2, "reports_to"),
    (2, 4, "reports_to"),
    (3, 1, "works_with"),
    (3, 4, "reports_to"),
    (5, 2, "works_with")
], ["src", "dst", "relationship"])

# Create graph
graph = GraphFrame(vertices, edges)

print("Vertices:")
graph.vertices.show()
print("\nEdges:")
graph.edges.show()

# Graph statistics
print("\n=== Graph Statistics ===")
print(f"Number of vertices: {graph.vertices.count()}")
print(f"Number of edges: {graph.edges.count()}")
print(f"In-degree: {graph.inDegrees.sort('inDegree', ascending=False).show()}")
print(f"Out-degree: {graph.outDegrees.sort('outDegree', ascending=False).show()}")

# Triangle counting (for undirected graphs)
print("\n=== Triangle Counting ===")
triangles = graph.triangleCount()
print("Triangle counts by vertex:")
triangles.sort('count', ascending=False).show()

# PageRank algorithm
print("\n=== PageRank ===")
ranks = graph.pageRank(resetProbability=0.15, maxIter=5)
print("PageRank scores:")
ranks.vertices.select("name", "pagerank").show()

# Breadth-first search
print("\n=== Breadth-First Search ===")
bfs_result = graph.bfs(fromExpr="name = 'Alice'", toExpr="name = 'David'")
print("Shortest path from Alice to David:")
bfs_result.show(truncate=False)

# Connected components
print("\n=== Connected Components ===")
components = graph.connectedComponents()
print("Connected components:")
components.select("id", "name", "component").sort("component").show()

# Find paths
print("\n=== Find Paths ===")
motifs = graph.find("(a)-[e]->(b)").filter("a.department = 'Engineering'")
print("Engineers and their connections:")
motifs.show()


## 15. Delta Lake and Data Quality

Implement ACID transactions with Delta Lake, perform data versioning, time travel, schema enforcement, and data quality checks.

### Delta Lake Features:
- **ACID Transactions**: Consistency and reliability
- **Schema Enforcement**: Prevent data corruption
- **Data Versioning**: Access historical versions
- **Time Travel**: Query data at specific points in time
- **Data Quality**: Constraints and checks
